<a href="https://colab.research.google.com/github/jsonjj/diagnostic-distractor-slm/blob/main/notebooks/train_qwen3_distractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diagnostic Distractor SLM — Qwen3-1.7B (QLoRA)

Fine-tune Qwen3-1.7B to generate 3 misconception-tagged, numerically-consistent distractors for middle-school "Number" MCQs.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Flow: install → clone repo → load base model → **litmus** (show the base model fails) → **QLoRA fine-tune** → **base-vs-tuned eval**.

In [1]:
%%capture
# Unsloth pulls a compatible torch/transformers/peft/trl/bitsandbytes/datasets stack.
!pip install -q unsloth
!pip install -q openai python-dotenv
# If this errors on Colab, use the official install snippet at https://unsloth.ai/docs

In [2]:
import os, sys

if not os.path.exists("diagnostic-distractor-slm"):
    !git clone https://github.com/jsonjj/diagnostic-distractor-slm.git
%cd diagnostic-distractor-slm
sys.path.insert(0, ".")

MODEL_NAME  = "unsloth/Qwen3-1.7B-bnb-4bit"
MAX_SEQ_LEN = 2048
EPOCHS      = 3

# Optional: TrueFoundry key, only needed if you later run the judge (`--judge`).
# from getpass import getpass; os.environ["TFY_API_KEY"] = getpass("TFY_API_KEY: ")

Cloning into 'diagnostic-distractor-slm'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 54 (delta 22), reused 51 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 210.49 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/diagnostic-distractor-slm


In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
print("loaded", MODEL_NAME)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.1: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

loaded unsloth/Qwen3-1.7B-bnb-4bit


## Litmus: can the *base* model already do it?

Run the base model on the held-out real questions and score it. We expect low spec-pass / alignment — the justification to fine-tune.

In [4]:
import json
from src.prompts import SYSTEM_PROMPT, build_user, parse_distractors

gold = [json.loads(l) for l in open("data/processed/eval_heldout.jsonl")]

def generate_json(model, tokenizer, system, user, max_new_tokens=512):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt",
    ).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def run_predictions(model, tokenizer, gold, out_path):
    FastLanguageModel.for_inference(model)
    rows = []
    for i, r in enumerate(gold):
        txt = generate_json(model, tokenizer, SYSTEM_PROMPT, build_user(r["question"], r["correct"], r["topic"]))
        rows.append({"id": r["id"], "distractors": parse_distractors(txt)})
        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(gold)}")
    with open(out_path, "w") as f:
        for x in rows:
            f.write(json.dumps(x) + "\n")
    return rows

run_predictions(model, tokenizer, gold, "predictions_base.jsonl")
print("\n===== BASE model =====")
!python -m src.eval predictions_base.jsonl

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

  20/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  40/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  100/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  120/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  140/140

===== BASE model =====
ALIGNMENT: {'Exact@3': 0.0, 'Partial@3': 35.0, 'Proportional@3': 12.4}
STRUCTURAL: {'exactly_3': 97.1, 'distinct_misconceptions': 91.4, 'none_equals_key': 62.1, 'distinct_answers': 70.7, 'spec_pass': 43.6}


## Fine-tune with QLoRA

LoRA on all linear layers, assistant-only loss, on the **v1 dataset — the chosen ship model** (1,341 examples: 141 real Eedi "Number" seed records + 1,200 unique, consistency-verified synthetic). We ship v1; the v2 rebalance (`train_v2.jsonl`) is kept in-repo as an alternative but is not the shipped run.

In [5]:
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

def to_text(ex):
    msgs = [
        {"role": "system", "content": ex["system"]},
        {"role": "user", "content": ex["user"]},
        {"role": "assistant", "content": ex["assistant"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)

ds = load_dataset("json", data_files="data/processed/train_v1.jsonl", split="train")  # v1 = ship model
ds = ds.map(lambda ex: {"text": to_text(ex)})

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=EPOCHS, learning_rate=2e-4,
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=42, output_dir="outputs", report_to="none",
    ),
)
# Train only on the assistant response (Qwen3 chat markers).
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

Unsloth 2026.7.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1341 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1341 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,341 | Num Epochs = 3 | Total steps = 504
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.213731
20,0.376331
30,0.308828
40,0.237810
50,0.217266
60,0.167839
70,0.190418
80,0.187465
90,0.173097
100,0.202790


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-504/tokenizer_config.json.


TrainOutput(global_step=504, training_loss=0.15068065889534496, metrics={'train_runtime': 1047.5995, 'train_samples_per_second': 3.84, 'train_steps_per_second': 0.481, 'total_flos': 1.08172717334016e+16, 'train_loss': 0.15068065889534496, 'epoch': 3.0})

## Evaluate the tuned model (base vs tuned)

In [6]:
run_predictions(model, tokenizer, gold, "predictions_tuned.jsonl")
print("\n===== TUNED model =====")
!python -m src.eval predictions_tuned.jsonl
print("\n===== BASE model (for comparison) =====")
!python -m src.eval predictions_base.jsonl

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=5

  20/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  40/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  100/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  120/140


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  140/140

===== TUNED model =====
ALIGNMENT: {'Exact@3': 12.1, 'Partial@3': 56.4, 'Proportional@3': 31.9}
STRUCTURAL: {'exactly_3': 100.0, 'distinct_misconceptions': 30.7, 'none_equals_key': 90.0, 'distinct_answers': 87.9, 'spec_pass': 81.4}

===== BASE model (for comparison) =====
ALIGNMENT: {'Exact@3': 0.0, 'Partial@3': 35.0, 'Proportional@3': 12.4}
STRUCTURAL: {'exactly_3': 97.1, 'distinct_misconceptions': 91.4, 'none_equals_key': 62.1, 'distinct_answers': 70.7, 'spec_pass': 43.6}


## Save the adapter (and optionally push to Hugging Face)

In [7]:
model.save_pretrained("qwen3-1.7b-distractor-lora")
tokenizer.save_pretrained("qwen3-1.7b-distractor-lora")
print("saved adapter -> qwen3-1.7b-distractor-lora/")

# Optional: push to Hugging Face (set your token first)
# from huggingface_hub import login; login(token="hf_...")
# model.push_to_hub("jsonjj/qwen3-1.7b-distractor-lora")
# tokenizer.push_to_hub("jsonjj/qwen3-1.7b-distractor-lora")

Unsloth: Restored added_tokens_decoder metadata in qwen3-1.7b-distractor-lora/tokenizer_config.json.


saved adapter -> qwen3-1.7b-distractor-lora/


In [8]:
# Download the prediction files so they can be handed back for the judged base-vs-v1 eval
# (feed predictions_base.jsonl / predictions_tuned.jsonl to `python -m src.eval ... --rubric`).
import os

_PRED_FILES = ["predictions_base.jsonl", "predictions_tuned.jsonl"]
try:
    from google.colab import files  # only present on Colab
    for _f in _PRED_FILES:
        if os.path.exists(_f):
            print(f"downloading {_f} ...")
            files.download(_f)
        else:
            print(f"skip (not found — run the eval cells first): {_f}")
except (ImportError, ModuleNotFoundError):
    # Not on Colab: nothing to download, just point at the files on disk.
    print("Not running on Colab — skipping auto-download.")
    for _f in _PRED_FILES:
        print(f"  {'found' if os.path.exists(_f) else 'missing'}: {_f}")

downloading predictions_base.jsonl ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloading predictions_tuned.jsonl ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Notes
- Expect **tuned spec-pass and alignment >> base** — that gap is the core result.
- Judge-based consistency (Claude Sonnet 5 via TrueFoundry) needs a key: `!python -m src.eval predictions_tuned.jsonl --judge` (costs API).
- If an Unsloth/TRL API differs on Colab, mirror the current Unsloth Qwen3 notebook at https://unsloth.ai/docs.